In [1]:
import re
import os

def refactor_sound_events(java_code: str) -> str:
    """
    Refatora o código Java de registro de sons para usar DeferredHolder e um padrão moderno.
    """
    # Regex para encontrar as linhas de registro de som antigas.
    # Captura o nome da variável (grupo 1) e o caminho do som (grupo 2).
    old_pattern = re.compile(
        r'^\s*public static final Supplier<SoundEvent>\s+([A-Z0-9_]+)\s*=\s*registerSoundEvent\("(.+?)"\);'
    )

    lines = java_code.split('\n')
    new_lines = []
    imports_added = False

    for line in lines:
        match = old_pattern.match(line)

        if match:
            # Encontrou uma linha de som para refatorar
            full_sound_path = match.group(2)

            # Extrai o nome do som do final do caminho e o converte para MAIÚSCULAS
            # Ex: "custom/.../dry_fire" -> "dry_fire" -> "DRY_FIRE"
            sound_name_lower = os.path.basename(full_sound_path)
            variable_name = sound_name_lower.upper()

            # Cria a nova linha refatorada
            new_line = (
                f'    public static final DeferredHolder<SoundEvent, SoundEvent> {variable_name} = SOUND_EVENTS.register("{sound_name_lower}",\n'
                f'            () -> SoundEvent.createVariableRangeEvent(new ResourceLocation(GunMod.MOD_ID, "{sound_name_lower}")));'
            )
            new_lines.append(new_line)

        else:
            # Mantém a linha original se não for uma declaração de som
            new_lines.append(line)
            
            # Adiciona a importação necessária do DeferredHolder
            if "import java.util.function.Supplier;" in line and not imports_added:
                new_lines.append("import net.neoforged.neoforge.registries.DeferredHolder;")
                imports_added = True
            
            # Remove a importação antiga se a nova já foi adicionada
            if "import java.util.function.Supplier;" in line and imports_added:
                 # Esta linha remove a importação do Supplier.
                 # Se você ainda usa Supplier em outro lugar, comente a linha abaixo.
                 new_lines.pop() # Remove a linha "import...Supplier" que acabou de ser adicionada
    
    return "\n".join(new_lines)

# --- CÉLULA PRINCIPAL DE EXECUÇÃO ---

# IMPORTANTE: Altere este caminho para o local exato do seu arquivo Java.
# Se o seu notebook estiver no mesmo diretório do projeto, o caminho pode ser relativo.
file_path = 'src/main/java/com/tacz/guns/init/ModSoundEvents.java' 

try:
    # Ler o conteúdo original do arquivo
    with open(file_path, 'r', encoding='utf-8') as f:
        original_code = f.read()

    # Executar a refatoração
    refactored_code = refactor_sound_events(original_code)

    # Mostrar o resultado para o usuário
    print("--- ARQUIVO ORIGINAL ---")
    print("```java")
    print(original_code)
    print("```")
    print("\n" + "="*80 + "\n")
    print("--- ARQUIVO REFATORADO ---")
    print("```java")
    print(refactored_code)
    print("```")
    print("\n" + "="*80 + "\n")

    # Pedir confirmação para salvar
    confirm = input(f"Deseja sobrescrever o arquivo '{file_path}' com as alterações acima? (s/N): ")

    if confirm.lower() == 's':
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(refactored_code)
        print(f"\n✅ Arquivo '{file_path}' foi atualizado com sucesso!")
    else:
        print("\n❌ Nenhuma alteração foi salva no arquivo.")

except FileNotFoundError:
    print(f"ERRO: O arquivo não foi encontrado em '{file_path}'.")
    print("Por favor, verifique se o caminho está correto e se o arquivo existe.")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

--- ARQUIVO ORIGINAL ---
```java
package com.tacz.guns.init;

import com.tacz.guns.GunMod;
import net.minecraft.core.registries.Registries;
import net.minecraft.resources.ResourceLocation;
import net.minecraft.sounds.SoundEvent;
import net.neoforged.neoforge.registries.DeferredRegister;
import java.util.function.Supplier;

public class ModSoundEvents {
    public static final DeferredRegister<SoundEvent> SOUND_EVENTS = DeferredRegister.create(Registries.SOUND_EVENT, GunMod.MOD_ID);

    public static final Supplier<SoundEvent> CUSTOM_TACZ_DEFAULT_GUN_ASSETS_TACZ_TACZ_SOUNDS_DRY_FIRE = registerSoundEvent("custom/tacz_default_gun/assets/tacz/tacz_sounds/dry_fire");
    public static final Supplier<SoundEvent> CUSTOM_TACZ_DEFAULT_GUN_ASSETS_TACZ_TACZ_SOUNDS_FIRE_SELECT = registerSoundEvent("custom/tacz_default_gun/assets/tacz/tacz_sounds/fire_select");
    public static final Supplier<SoundEvent> CUSTOM_TACZ_DEFAULT_GUN_ASSETS_TACZ_TACZ_SOUNDS_FLESH_HIT = registerSoundEvent("custom/tacz_d

In [3]:
import re
import os

def refactor_sound_events_custom_path(java_code: str) -> str:
    """
    Refatora o código Java para usar DeferredHolder, mantendo os nomes de 
    variáveis curtos, mas usando o caminho completo como ID de registro do som.
    """
    # Regex para encontrar as linhas de registro de som antigas.
    # Captura o caminho completo do som dentro de registerSoundEvent("...").
    old_pattern = re.compile(
        r'^\s*public static final Supplier<SoundEvent>\s+[A-Z0-9_]+\s*=\s*registerSoundEvent\("(.+?)"\);'
    )

    lines = java_code.split('\n')
    new_lines = []
    imports_added = False
    
    # Encontra o ponto para remover a importação antiga
    has_supplier_import = "import java.util.function.Supplier;" in java_code

    for line in lines:
        match = old_pattern.match(line)

        if match:
            # Encontrou uma linha de som para refatorar
            full_sound_path = match.group(1) # Captura o caminho completo, ex: "custom/.../dry_fire"

            # Extrai o nome CURTO do final do caminho APENAS para o NOME DA VARIÁVEL JAVA
            # Ex: "custom/.../dry_fire" -> "dry_fire"
            sound_name_lower = os.path.basename(full_sound_path)
            # Converte para MAIÚSCULAS para o nome da variável, ex: "DRY_FIRE"
            variable_name = sound_name_lower.upper().replace('.', '_')

            # Cria a nova linha refatorada, usando o CAMINHO COMPLETO como ID de registro
            new_line = (
                f'    public static final DeferredHolder<SoundEvent, SoundEvent> {variable_name} = SOUND_EVENTS.register("{full_sound_path}",\n'
                f'            () -> SoundEvent.createVariableRangeEvent(ResourceLocation.fromNamespaceAndPath(GunMod.MOD_ID, "{full_sound_path}")));'
            )
            new_lines.append(new_line)

        else:
            # Lógica para adicionar/remover imports
            if "import java.util.function.Supplier;" in line:
                if not imports_added:
                    # Adiciona a nova importação logo abaixo da antiga
                    new_lines.append(line)
                    new_lines.append("import net.neoforged.neoforge.registries.DeferredHolder;")
                    imports_added = True
                # Se a importação já foi adicionada, esta linha (a do Supplier) é pulada,
                # efetivamente a removendo.
            else:
                 new_lines.append(line)

    return "\n".join(new_lines)

# --- CÉLULA PRINCIPAL DE EXECUÇÃO ---

# O arquivo de entrada agora é o mod_sound_events_output.txt
file_path = 'mod_sound_events_output.txt' 

try:
    # Ler o conteúdo original do arquivo
    with open(file_path, 'r', encoding='utf-8') as f:
        original_code = f.read()

    # Executar a refatoração
    refactored_code = refactor_sound_events_custom_path(original_code)

    # Mostrar o resultado para o usuário
    print("--- ARQUIVO DE ENTRADA (mod_sound_events_output.txt) ---")
    print("```java")
    print(original_code)
    print("```")
    print("\n" + "="*80 + "\n")
    print("--- ARQUIVO REFATORADO (RESULTADO) ---")
    print("```java")
    print(refactored_code)
    print("```")
    print("\n" + "="*80 + "\n")

    # Pedir confirmação para salvar
    confirm = input(f"Deseja sobrescrever o arquivo '{file_path}' com as alterações acima? (s/N): ")

    if confirm.lower() == 's':
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(refactored_code)
        print(f"\n✅ Arquivo '{file_path}' foi atualizado com sucesso!")
    else:
        print("\n❌ Nenhuma alteração foi salva no arquivo.")

except FileNotFoundError:
    print(f"ERRO: O arquivo não foi encontrado em '{file_path}'.")
    print("Por favor, verifique se o arquivo existe no mesmo diretório que o seu notebook, ou forneça o caminho completo.")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

--- ARQUIVO DE ENTRADA (mod_sound_events_output.txt) ---
```java
package com.tacz.guns.init;

import com.tacz.guns.GunMod;
import net.minecraft.core.registries.Registries;
import net.minecraft.resources.ResourceLocation;
import net.minecraft.sounds.SoundEvent;
import net.neoforged.neoforge.registries.DeferredRegister;
import java.util.function.Supplier;

public class ModSoundEvents {
    public static final DeferredRegister<SoundEvent> SOUND_EVENTS = DeferredRegister.create(Registries.SOUND_EVENT, GunMod.MOD_ID);

    public static final Supplier<SoundEvent> CUSTOM_TACZ_DEFAULT_GUN_ASSETS_TACZ_TACZ_SOUNDS_DRY_FIRE = registerSoundEvent("custom/tacz_default_gun/assets/tacz/tacz_sounds/dry_fire");
    public static final Supplier<SoundEvent> CUSTOM_TACZ_DEFAULT_GUN_ASSETS_TACZ_TACZ_SOUNDS_FIRE_SELECT = registerSoundEvent("custom/tacz_default_gun/assets/tacz/tacz_sounds/fire_select");
    public static final Supplier<SoundEvent> CUSTOM_TACZ_DEFAULT_GUN_ASSETS_TACZ_TACZ_SOUNDS_FLESH_HIT = r